# Entraînement Baseline MobileNetV2 - Google Colab

Ce notebook permet d'entraîner le modèle baseline sur Google Colab pour accélérer l'entraînement avec GPU.

## Prérequis
1. Dataset préparé (train/val/test splits)
2. Runtime Colab avec GPU activé (Runtime > Change runtime type > GPU)

## Workflow
1. Installation des dépendances
2. Upload du dataset
3. Vérification de la structure
4. Entraînement du modèle
5. Téléchargement des résultats

## 1. Vérification du GPU

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
else:
    print("⚠️ WARNING: GPU not available. Training will be slow!")
    print("Enable GPU: Runtime > Change runtime type > GPU")

## 2. Installation des dépendances

Installation des packages nécessaires pour l'entraînement.

In [ ]:
# Installation des dépendances principales
!pip install -q torch torchvision tqdm matplotlib pillow

## 3. Upload du Dataset

### Option A: Upload manuel (recommandé pour dataset < 500 MB)

**Structure requise pour le ZIP:**
```
prepared.zip
├── train/
│   ├── safe/       (699 images .jpg)
│   └── dangerous/  (699 images .jpg)
├── val/
│   ├── safe/       (147 images .jpg)
│   └── dangerous/  (147 images .jpg)
└── test/
    ├── safe/       (102 images .jpg)
    └── dangerous/  (102 images .jpg)
```

**Instructions:**
1. Depuis votre projet local, compresser le dossier `data/prepared/` en ZIP
2. Nommer le fichier `prepared.zip`
3. Exécuter la cellule ci-dessous et sélectionner le fichier

In [ ]:
from google.colab import files
import zipfile
import os

print("📤 Upload du fichier prepared.zip...")
uploaded = files.upload()

# Extraire le ZIP
print("\n📦 Extraction du dataset...")
with zipfile.ZipFile('prepared.zip', 'r') as zip_ref:
    zip_ref.extractall('data')

print("\n✅ Dataset extrait dans /content/data/prepared/")

### Option B: Upload depuis Google Drive (recommandé pour dataset > 500 MB)

**Instructions:**
1. Uploader `prepared.zip` dans votre Google Drive
2. Exécuter la cellule ci-dessous
3. Autoriser l'accès à Google Drive
4. Modifier le chemin si nécessaire

In [ ]:
from google.colab import drive
import zipfile
import shutil

# Monter Google Drive
drive.mount('/content/drive')

# Copier le ZIP depuis Drive (modifier le chemin si nécessaire)
drive_zip_path = '/content/drive/MyDrive/prepared.zip'  # Modifier ce chemin

if os.path.exists(drive_zip_path):
    print(f"📋 Copie depuis Drive: {drive_zip_path}")
    shutil.copy(drive_zip_path, '/content/prepared.zip')
    
    # Extraire
    print("📦 Extraction du dataset...")
    with zipfile.ZipFile('/content/prepared.zip', 'r') as zip_ref:
        zip_ref.extractall('data')
    
    print("✅ Dataset extrait!")
else:
    print(f"❌ Fichier non trouvé: {drive_zip_path}")
    print("Vérifiez le chemin dans Google Drive")

## 4. Vérification de la structure du dataset

In [ ]:
import os
from pathlib import Path

def verify_dataset_structure(base_path='data/prepared'):
    """Vérifie que la structure du dataset est correcte"""
    base = Path(base_path)
    
    if not base.exists():
        print(f"❌ Dossier {base_path} non trouvé!")
        return False
    
    print("📂 Structure du dataset:\n")
    
    total_images = 0
    for split in ['train', 'val', 'test']:
        split_path = base / split
        if not split_path.exists():
            print(f"❌ {split}/ manquant!")
            continue
            
        safe_count = len(list((split_path / 'safe').glob('*.jpg'))) if (split_path / 'safe').exists() else 0
        dangerous_count = len(list((split_path / 'dangerous').glob('*.jpg'))) if (split_path / 'dangerous').exists() else 0
        split_total = safe_count + dangerous_count
        total_images += split_total
        
        print(f"  {split}/")
        print(f"    safe: {safe_count} images")
        print(f"    dangerous: {dangerous_count} images")
        print(f"    Total: {split_total} images\n")
    
    print(f"\n✅ Total: {total_images} images")
    
    if total_images == 0:
        print("\n⚠️ ATTENTION: Aucune image trouvée!")
        return False
    
    return True

# Vérifier la structure
if verify_dataset_structure():
    print("\n🎉 Dataset correctement structuré!")
else:
    print("\n❌ Problème avec la structure du dataset.")
    print("Vérifiez que le ZIP contient bien train/val/test avec safe/ et dangerous/")

## 5. Code d'entraînement

Définition des classes et fonctions pour l'entraînement.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from PIL import Image
import os
import json
from tqdm import tqdm
import matplotlib.pyplot as plt
from typing import Tuple, Dict, List
import numpy as np
from pathlib import Path


class DangerousObjectDataset(Dataset):
    """Dataset pour les images safe/dangerous"""

    def __init__(self, data_dir: str, split: str, transform=None):
        """
        Args:
            data_dir: Chemin vers data/prepared/
            split: 'train', 'val' ou 'test'
            transform: Transformations à appliquer
        """
        self.data_dir = Path(data_dir)
        self.split = split
        self.transform = transform

        self.images = []
        self.labels = []

        # Charger les images depuis safe/ et dangerous/
        split_dir = self.data_dir / split

        # Classe safe (label 0)
        safe_dir = split_dir / 'safe'
        if safe_dir.exists():
            for img_path in safe_dir.glob('*.jpg'):
                self.images.append(str(img_path))
                self.labels.append(0)

        # Classe dangerous (label 1)
        dangerous_dir = split_dir / 'dangerous'
        if dangerous_dir.exists():
            for img_path in dangerous_dir.glob('*.jpg'):
                self.images.append(str(img_path))
                self.labels.append(1)

        print(f"  {split}: {len(self.images)} images "
              f"(safe: {self.labels.count(0)}, dangerous: {self.labels.count(1)})")

    def __len__(self) -> int:
        return len(self.images)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int]:
        img_path = self.images[idx]
        label = self.labels[idx]

        # Charger l'image
        image = Image.open(img_path).convert('RGB')

        # Appliquer les transformations
        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
class MobileNetV2Classifier(nn.Module):
    """Classificateur basé sur MobileNetV2"""

    def __init__(self, num_classes: int = 2, pretrained: bool = True):
        super().__init__()

        # Charger MobileNetV2 pré-entraîné sur ImageNet
        self.mobilenet = models.mobilenet_v2(pretrained=pretrained)

        # Freeze early layers (transfer learning)
        for param in self.mobilenet.features[:-3].parameters():
            param.requires_grad = False

        # Remplacer le classifier
        in_features = self.mobilenet.classifier[1].in_features

        self.mobilenet.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.mobilenet(x)

In [ ]:
class EarlyStopping:
    """Early stopping pour éviter l'overfitting"""

    def __init__(self, patience: int = 7, min_delta: float = 0.0, verbose: bool = True):
        self.patience = patience
        self.min_delta = min_delta
        self.verbose = verbose
        self.counter = 0
        self.best_loss = None
        self.early_stop = False
        self.best_model_state = None

    def __call__(self, val_loss: float, model: nn.Module) -> bool:
        if self.best_loss is None:
            self.best_loss = val_loss
            self.best_model_state = model.state_dict().copy()
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.verbose:
                print(f"  EarlyStopping counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
                if self.verbose:
                    print("  Early stopping triggered!")
        else:
            if self.verbose:
                print(f"  Validation loss improved: {self.best_loss:.4f} → {val_loss:.4f}")
            self.best_loss = val_loss
            self.best_model_state = model.state_dict().copy()
            self.counter = 0

        return self.early_stop

## 6. Configuration et lancement de l'entraînement

In [ ]:
# Configuration
DATA_DIR = 'data/prepared'
OUTPUT_DIR = 'models/baseline'
BATCH_SIZE = 32
NUM_EPOCHS = 50
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-4

# Créer le dossier de sortie
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Device: {device}")

# Transformations
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                       std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                       std=[0.229, 0.224, 0.225])
])

In [ ]:
# Créer les datasets
print("📦 Chargement des datasets...\n")

train_dataset = DangerousObjectDataset(DATA_DIR, 'train', train_transform)
val_dataset = DangerousObjectDataset(DATA_DIR, 'val', val_transform)
test_dataset = DangerousObjectDataset(DATA_DIR, 'test', val_transform)

# Créer les dataloaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"\n✅ Dataloaders créés!")

In [ ]:
# Créer le modèle
print("\n🏗️ Création du modèle MobileNetV2...")
model = MobileNetV2Classifier(num_classes=2, pretrained=True)
model = model.to(device)

# Compter les paramètres
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")

# Loss et optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5
)

# Early stopping
early_stopping = EarlyStopping(patience=10, verbose=True)

In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, device):
    """Entraîne le modèle pour une epoch"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(train_loader, desc="Training")
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        pbar.set_postfix({'loss': loss.item(), 'acc': 100. * correct / total})

    epoch_loss = running_loss / total
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc


def validate(model, val_loader, criterion, device):
    """Évalue le modèle sur le validation set"""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc="Validation"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc

In [ ]:
# Entraînement
print("="*60)
print("🚀 ENTRAÎNEMENT BASELINE MOBILENETV2")
print("="*60)

history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': []
}

best_val_acc = 0.0

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\nEpoch {epoch}/{NUM_EPOCHS}")
    print("-" * 40)

    # Train
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)

    # Validate
    val_loss, val_acc = validate(model, val_loader, criterion, device)

    # Update learning rate
    scheduler.step(val_loss)

    # Sauvegarder l'historique
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    # Afficher les résultats
    print(f"\n📊 Résultats Epoch {epoch}:")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"  Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.2f}%")

    # Sauvegarder le meilleur modèle
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'val_loss': val_loss,
        }, os.path.join(OUTPUT_DIR, 'best_model.pth'))
        print(f"  ✓ Meilleur modèle sauvegardé (val_acc: {val_acc:.2f}%)")

    # Early stopping
    if early_stopping(val_loss, model):
        print(f"\n⏹️ Early stopping à l'epoch {epoch}")
        model.load_state_dict(early_stopping.best_model_state)
        break

print("\n" + "="*60)
print("✅ ENTRAÎNEMENT TERMINÉ")
print("="*60)

## 7. Évaluation finale sur le test set

In [ ]:
print("="*60)
print("🧪 ÉVALUATION FINALE SUR TEST SET")
print("="*60)

test_loss, test_acc = validate(model, test_loader, criterion, device)
print(f"\n📊 Résultats Test:")
print(f"  Test Loss: {test_loss:.4f}")
print(f"  Test Acc:  {test_acc:.2f}%")

# Sauvegarder le modèle final
torch.save({
    'model_state_dict': model.state_dict(),
    'test_acc': test_acc,
    'test_loss': test_loss,
    'history': history
}, os.path.join(OUTPUT_DIR, 'final_model.pth'))

# Sauvegarder l'historique
with open(os.path.join(OUTPUT_DIR, 'training_history.json'), 'w') as f:
    json.dump(history, f, indent=2)

print(f"\n🎯 Meilleure validation accuracy: {best_val_acc:.2f}%")
print(f"🧪 Test accuracy finale: {test_acc:.2f}%")

## 8. Visualisation des résultats

In [ ]:
# Générer les graphiques
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, len(history['train_loss']) + 1)

# Loss
ax1.plot(epochs, history['train_loss'], 'b-', label='Train Loss', linewidth=2)
ax1.plot(epochs, history['val_loss'], 'r-', label='Val Loss', linewidth=2)
ax1.set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Accuracy
ax2.plot(epochs, history['train_acc'], 'b-', label='Train Acc', linewidth=2)
ax2.plot(epochs, history['val_acc'], 'r-', label='Val Acc', linewidth=2)
ax2.set_title('Training and Validation Accuracy', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy (%)', fontsize=12)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_history.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📈 Graphiques sauvegardés: {OUTPUT_DIR}/training_history.png")

## 9. Téléchargement des résultats

Téléchargez les fichiers suivants pour les importer dans votre projet local:
- `best_model.pth` - Meilleur modèle (validation accuracy)
- `final_model.pth` - Modèle final après entraînement
- `training_history.json` - Historique d'entraînement
- `training_history.png` - Graphiques de performance

In [ ]:
import shutil
from google.colab import files

# Créer un ZIP avec tous les résultats
print("📦 Création de l'archive des résultats...")
shutil.make_archive('baseline_results', 'zip', OUTPUT_DIR)

print("\n📊 Contenu de l'archive:")
print("  - best_model.pth")
print("  - final_model.pth")
print("  - training_history.json")
print("  - training_history.png")

# Télécharger l'archive
print("\n⬇️ Téléchargement de baseline_results.zip...")
files.download('baseline_results.zip')

print("\n✅ Téléchargement terminé!")
print("\n📂 Pour importer dans votre projet:")
print("   1. Extraire baseline_results.zip")
print("   2. Copier le contenu dans models/baseline/")

## 10. Résumé des résultats

In [ ]:
print("="*60)
print("📋 RÉSUMÉ DE L'ENTRAÎNEMENT")
print("="*60)
print(f"\n🏗️ Architecture: MobileNetV2")
print(f"📊 Dataset:")
print(f"   - Train: {len(train_dataset)} images")
print(f"   - Val:   {len(val_dataset)} images")
print(f"   - Test:  {len(test_dataset)} images")
print(f"\n⚙️ Hyperparamètres:")
print(f"   - Epochs: {epoch} (early stopped)")
print(f"   - Batch size: {BATCH_SIZE}")
print(f"   - Learning rate: {LEARNING_RATE}")
print(f"   - Weight decay: {WEIGHT_DECAY}")
print(f"\n🎯 Performance:")
print(f"   - Meilleure val accuracy: {best_val_acc:.2f}%")
print(f"   - Test accuracy finale:  {test_acc:.2f}%")
print(f"   - Test loss finale:      {test_loss:.4f}")
print(f"\n🖥️ Device utilisé: {device}")
print("="*60)

## Prochaines étapes

1. ✅ Télécharger `baseline_results.zip`
2. Extraire le contenu dans votre projet local: `models/baseline/`
3. Vérifier les résultats avec les graphiques
4. Utiliser le modèle pour l'évaluation ou les tests adversariaux

---

### Conseils
- Si la performance est insuffisante, essayez d'augmenter le nombre d'epochs
- Pour améliorer les résultats, ajustez le learning rate ou le weight decay
- Consultez les graphiques pour détecter l'overfitting
- Le modèle est sauvegardé automatiquement à chaque amélioration